
# EF–ToM band-ERP, PSD and DE feature extraction

Start from the **interpolated epoch files**. This notebook extracts three families of participant-level EEG features:

1. **Band-limited evoked/ERP features** in theta, alpha, low-beta and beta.
2. **PSD** by condition × band × ROI.
3. **Differential entropy (DE)** by condition × band × ROI.

PSD/DE are kept **condition-specific** rather than automatically subtracting conditions. EF and ToM are processed independently, and ToM hearing/thinking are kept separate.


In [2]:

from pathlib import Path
import numpy as np
import pandas as pd
import mne
from scipy.signal import welch

mne.set_log_level("WARNING")


## 1. Paths and feature definitions

In [3]:

BASE = Path(r"H:\ASD projoect")

# Change these for one matched participant
GROUP = "ASD"
EF_ID = "1170"
TOM_ID = "3200"

EF_DIR = BASE / "EF" / GROUP / f"{EF_ID}_CF_CV_processed_numpy"
TOM_DIR = BASE / "ToM" / GROUP / f"{TOM_ID}a_CV_processed_numpy"

EF_FILE = EF_DIR / "interpolated_epoch.npy"
TOM_HEARING_FILE = TOM_DIR / "interpolated_hearing_epoch.npy"
TOM_THINKING_FILE = TOM_DIR / "interpolated_thinking_epoch.npy"

BANDS = {
    "theta": (4, 7),
    "alpha": (8, 12),
    "low_beta": (12, 16),
    "beta": (16, 25),
}

ROIS = {
    "frontal": ["FP1","FPZ","FP2","AF3","AF4","F7","F5","F3","F1","FZ","F2","F4","F6","F8"],
    "frontocentral": ["FT7","FC5","FC3","FC1","FCZ","FC2","FC4","FC6","FT8"],
    "central": ["T7","C5","C3","C1","CZ","C2","C4","C6","T8"],
    "centroparietal": ["TP7","CP5","CP3","CP1","CPZ","CP2","CP4","CP6","TP8"],
    "parietal": ["P7","P5","P3","P1","PZ","P2","P4","P6","P8"],
    "posterior": ["PO7","PO5","PO3","POZ","PO4","PO6","PO8","O1","OZ","O2"],
}

print(EF_FILE)
print(TOM_HEARING_FILE)
print(TOM_THINKING_FILE)


H:\ASD projoect\EF\ASD\1170_CF_CV_processed_numpy\interpolated_epoch.npy
H:\ASD projoect\ToM\ASD\3200a_CV_processed_numpy\interpolated_hearing_epoch.npy
H:\ASD projoect\ToM\ASD\3200a_CV_processed_numpy\interpolated_thinking_epoch.npy


## 2. Load interpolated epochs

In [4]:

def load_interpolated(file_path):
    d = np.load(file_path, allow_pickle=True).item()
    meta = pd.DataFrame(d["metadata"])

    info = mne.create_info(
        ch_names=list(d["channels"]),
        sfreq=float(d["sfreq"]),
        ch_types="eeg",
    )
    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="ignore")

    ep = mne.EpochsArray(
        d["eeg"], info,
        tmin=float(d["times"][0]),
        metadata=meta,
        verbose=False,
    )
    ep.set_eeg_reference("average", projection=False, verbose=False)
    return ep


def get_conditions(epochs, task):
    m = epochs.metadata

    if task == "EF":
        g = m["go_nogo"].astype(str).str.lower()
        return {
            "go": (g == "go").to_numpy(),
            "nogo": (g == "nogo").to_numpy(),
        }

    if task == "ToM":
        tom = m["tom"].astype(str).str.lower()
        belief = m["belief"].astype(str).str.lower()
        return {
            "false_cog": ((belief == "false") & (tom == "cognitive")).to_numpy(),
            "true_cog":  ((belief == "true")  & (tom == "cognitive")).to_numpy(),
            "false_aff": ((belief == "false") & (tom == "affective")).to_numpy(),
            "true_aff":  ((belief == "true")  & (tom == "affective")).to_numpy(),
        }

    raise ValueError("task must be EF or ToM")



## 3. Band-limited evoked / ERP features

The same established component windows are retained. Because a signed mean of a narrow-band oscillation can cancel toward zero, the notebook saves **RMS**, **mean absolute amplitude**, and **absolute peak amplitude** of the band-limited evoked response.


In [5]:

def get_erp_defs(task):
    if task == "EF":
        return {
            "N2":      {"channels": ["FCZ"],       "tmin": 0.300, "tmax": 0.400},
            "P3_LPP":  {"channels": ["CZ","CPZ"], "tmin": 0.500, "tmax": 0.700},
        }

    posterior = ["CPZ","CP1","CP2","CP3","CP4","PZ","P1","P2","P3","P4","POZ","PO3","PO4"]
    return {
        "LPC":       {"channels": ["FZ","F1","F2","FCZ","FC1","FC2"], "tmin": 0.300, "tmax": 0.600},
        "early_LSW": {"channels": posterior, "tmin": 0.800, "tmax": 1.400},
        "mid_LSW":   {"channels": posterior, "tmin": 1.400, "tmax": 2.100},
        "late_LSW":  {"channels": posterior, "tmin": 2.000, "tmax": 2.500},
    }


def band_erp_features(epochs, task):
    conds = get_conditions(epochs, task)
    defs = get_erp_defs(task)
    rows = []

    for band, (lo, hi) in BANDS.items():
        bep = epochs.copy().filter(
            lo, hi,
            method="iir",
            iir_params=dict(order=4, ftype="butter"),
            verbose=False,
        )

        for cond, mask in conds.items():
            idx = np.where(mask)[0]
            if len(idx) == 0:
                continue

            evoked = bep[idx].average()

            for comp, cfg in defs.items():
                chs = [ch for ch in cfg["channels"] if ch in evoked.ch_names]
                if not chs:
                    continue

                x = (evoked.copy().pick(chs).crop(cfg["tmin"], cfg["tmax"]).data * 1e6)

                rows.append({
                    "task": task,
                    "condition": cond,
                    "band": band,
                    "component": comp,
                    "n_trials": len(idx),
                    "evoked_rms_uv": float(np.sqrt(np.mean(x**2))),
                    "evoked_mean_abs_uv": float(np.mean(np.abs(x))),
                    "evoked_peak_abs_uv": float(np.max(np.abs(x))),
                })

    return pd.DataFrame(rows)



## 4. PSD + DE

PSD and DE are computed **per trial**, then averaged within condition.

- EF: Go and Nogo are retained separately.
- ToM: false-cognitive, true-cognitive, false-affective, true-affective are retained separately.
- No subtraction is forced at this stage.

DE is computed after band-pass filtering:

\[
DE=
rac12\ln(2\pi e\sigma^2)
\]

The DE signal is converted to µV before variance estimation so every participant uses the same numerical scale.


In [6]:

def roi_indices(ch_names):
    out = {}
    for roi, roi_chs in ROIS.items():
        idx = [ch_names.index(ch) for ch in roi_chs if ch in ch_names]
        if idx:
            out[roi] = idx
    return out


def analysis_interval(epochs, task):
    if task == "EF":
        return 0.0, min(1.0, float(epochs.times[-1]))
    if task == "ToM":
        return 0.0, min(2.5, float(epochs.times[-1]))
    raise ValueError


def psd_de_trial_features(epochs, task):
    conds = get_conditions(epochs, task)
    tmin, tmax = analysis_interval(epochs, task)
    ep = epochs.copy().crop(tmin=tmin, tmax=tmax)

    x_v = ep.get_data()              # V
    x_uv = x_v * 1e6                 # µV for DE
    sf = float(ep.info["sfreq"])
    rois = roi_indices(ep.ch_names)

    labels = np.array(["unassigned"] * len(ep), dtype=object)
    for cond, mask in conds.items():
        labels[mask] = cond

    nperseg = min(int(round(sf)), x_v.shape[-1])
    noverlap = nperseg // 2 if nperseg >= 4 else 0

    freqs, psd = welch(
        x_v,
        fs=sf,
        axis=-1,
        nperseg=nperseg,
        noverlap=noverlap,
        detrend="constant",
    )

    rows = []
    eps = 1e-30

    for band, (lo, hi) in BANDS.items():
        fm = (freqs >= lo) & (freqs <= hi)
        if not np.any(fm):
            continue

        # trial x channel
        bp = psd[..., fm].mean(axis=-1)

        bx = mne.filter.filter_data(
            x_uv,
            sfreq=sf,
            l_freq=lo,
            h_freq=hi,
            method="iir",
            iir_params=dict(order=4, ftype="butter"),
            verbose=False,
        )
        var = np.var(bx, axis=-1, ddof=1)
        de = 0.5 * np.log(2 * np.pi * np.e * np.maximum(var, eps))

        for trial in range(len(ep)):
            cond = labels[trial]
            if cond == "unassigned":
                continue

            for roi, idx in rois.items():
                roi_psd = float(np.mean(bp[trial, idx]))
                roi_de = float(np.mean(de[trial, idx]))

                rows.append({
                    "task": task,
                    "trial_idx": trial,
                    "condition": cond,
                    "band": band,
                    "roi": roi,
                    "psd_db": 10 * np.log10(max(roi_psd, eps)),
                    "de": roi_de,
                })

    return pd.DataFrame(rows)


def summarize_psd_de(trial_df):
    return (
        trial_df
        .groupby(["task","condition","band","roi"], as_index=False)
        .agg(
            psd_db=("psd_db","mean"),
            de=("de","mean"),
            n_trials=("trial_idx","nunique"),
        )
    )


## 5. Helpers for wide CCA-ready feature rows

In [7]:

def band_erp_to_wide(df, prefix):
    row = {}
    for _, r in df.iterrows():
        base = f"{prefix}_bandERP_{r['condition']}_{r['band']}_{r['component']}"
        row[f"{base}_RMS_uV"] = r["evoked_rms_uv"]
        row[f"{base}_MeanAbs_uV"] = r["evoked_mean_abs_uv"]
        row[f"{base}_PeakAbs_uV"] = r["evoked_peak_abs_uv"]
    return row


def psd_de_to_wide(df, prefix):
    row = {}
    for _, r in df.iterrows():
        base = f"{prefix}_{r['condition']}_{r['band']}_{r['roi']}"
        row[f"{base}_PSD_dB"] = r["psd_db"]
        row[f"{base}_DE"] = r["de"]
    return row


## 6. Run one matched participant

In [8]:

# ----- EF -----
ef_epochs = load_interpolated(EF_FILE)
ef_band_erp = band_erp_features(ef_epochs, "EF")
ef_trial_pd = psd_de_trial_features(ef_epochs, "EF")
ef_pd = summarize_psd_de(ef_trial_pd)

# ----- ToM hearing -----
h_epochs = load_interpolated(TOM_HEARING_FILE)
h_band_erp = band_erp_features(h_epochs, "ToM")
h_trial_pd = psd_de_trial_features(h_epochs, "ToM")
h_pd = summarize_psd_de(h_trial_pd)

# ----- ToM thinking -----
t_epochs = load_interpolated(TOM_THINKING_FILE)
t_band_erp = band_erp_features(t_epochs, "ToM")
t_trial_pd = psd_de_trial_features(t_epochs, "ToM")
t_pd = summarize_psd_de(t_trial_pd)

print("EF band ERP:", ef_band_erp.shape)
print("EF PSD/DE:", ef_pd.shape)
print("Hearing band ERP:", h_band_erp.shape)
print("Hearing PSD/DE:", h_pd.shape)
print("Thinking band ERP:", t_band_erp.shape)
print("Thinking PSD/DE:", t_pd.shape)


EF band ERP: (16, 8)
EF PSD/DE: (48, 7)
Hearing band ERP: (64, 8)
Hearing PSD/DE: (96, 7)
Thinking band ERP: (64, 8)
Thinking PSD/DE: (96, 7)


## 7. Save one-participant outputs

In [9]:

# EF
ef_band_erp.to_csv(EF_DIR / "EF_bandERP_features.csv", index=False)
ef_trial_pd.to_csv(EF_DIR / "EF_PSD_DE_trial_features.csv", index=False)
ef_pd.to_csv(EF_DIR / "EF_PSD_DE_condition_features.csv", index=False)

# ToM hearing
h_band_erp.to_csv(TOM_DIR / "ToM_hearing_bandERP_features.csv", index=False)
h_trial_pd.to_csv(TOM_DIR / "ToM_hearing_PSD_DE_trial_features.csv", index=False)
h_pd.to_csv(TOM_DIR / "ToM_hearing_PSD_DE_condition_features.csv", index=False)

# ToM thinking
t_band_erp.to_csv(TOM_DIR / "ToM_thinking_bandERP_features.csv", index=False)
t_trial_pd.to_csv(TOM_DIR / "ToM_thinking_PSD_DE_trial_features.csv", index=False)
t_pd.to_csv(TOM_DIR / "ToM_thinking_PSD_DE_condition_features.csv", index=False)

row = {
    "group": GROUP,
    "EF_ID": EF_ID,
    "ToM_ID": TOM_ID,
    **band_erp_to_wide(ef_band_erp, "EF"),
    **psd_de_to_wide(ef_pd, "EF"),
    **band_erp_to_wide(h_band_erp, "H"),
    **psd_de_to_wide(h_pd, "H"),
    **band_erp_to_wide(t_band_erp, "T"),
    **psd_de_to_wide(t_pd, "T"),
}

wide = pd.DataFrame([row])
wide.to_csv(BASE / f"{GROUP}_{EF_ID}_{TOM_ID}_band_PSD_DE_features.csv", index=False)

print("Saved. Total columns:", wide.shape[1])


Saved. Total columns: 915


## 8. Optional batch processing for the currently matched participants

In [10]:

ASD_PAIRS = [
    ("1170","3200"),("1161","3081"),("1121","3041"),("1101","3061"),
    ("1141","3161"),("1151","3071"),("1191","3111"),("1031","3121"),
    ("1021","3021"),("1011","3011"),("1071","3051"),
]

TD_PAIRS = [
    ("2360","4150"),("2121","4181"),("2201","4171"),("2250","4210"),
    ("2281","4091"),("2340","4080"),("2151","4071"),("2211","4141"),
]


def extract_one_file(file_path, task, prefix):
    ep = load_interpolated(file_path)
    be = band_erp_features(ep, task)
    tr = psd_de_trial_features(ep, task)
    pd_sum = summarize_psd_de(tr)
    wide = {
        **band_erp_to_wide(be, prefix),
        **psd_de_to_wide(pd_sum, prefix),
    }
    return be, tr, pd_sum, wide


def process_group(group, pair_list):
    rows = []

    for ef_id, tom_id in pair_list:
        print(f"{group}: EF {ef_id} / ToM {tom_id}")

        ef_dir = BASE / "EF" / group / f"{ef_id}_CF_CV_processed_numpy"
        tom_dir = BASE / "ToM" / group / f"{tom_id}a_CV_processed_numpy"

        ef_file = ef_dir / "interpolated_epoch.npy"
        h_file = tom_dir / "interpolated_hearing_epoch.npy"
        t_file = tom_dir / "interpolated_thinking_epoch.npy"

        if not (ef_file.exists() and h_file.exists() and t_file.exists()):
            print("  missing interpolated file -> skipped")
            continue

        ef_be, ef_tr, ef_pd, ef_w = extract_one_file(ef_file, "EF", "EF")
        h_be, h_tr, h_pd, h_w = extract_one_file(h_file, "ToM", "H")
        t_be, t_tr, t_pd, t_w = extract_one_file(t_file, "ToM", "T")

        ef_be.to_csv(ef_dir / "EF_bandERP_features.csv", index=False)
        ef_tr.to_csv(ef_dir / "EF_PSD_DE_trial_features.csv", index=False)
        ef_pd.to_csv(ef_dir / "EF_PSD_DE_condition_features.csv", index=False)

        h_be.to_csv(tom_dir / "ToM_hearing_bandERP_features.csv", index=False)
        h_tr.to_csv(tom_dir / "ToM_hearing_PSD_DE_trial_features.csv", index=False)
        h_pd.to_csv(tom_dir / "ToM_hearing_PSD_DE_condition_features.csv", index=False)

        t_be.to_csv(tom_dir / "ToM_thinking_bandERP_features.csv", index=False)
        t_tr.to_csv(tom_dir / "ToM_thinking_PSD_DE_trial_features.csv", index=False)
        t_pd.to_csv(tom_dir / "ToM_thinking_PSD_DE_condition_features.csv", index=False)

        rows.append({
            "group": group,
            "EF_ID": ef_id,
            "ToM_ID": tom_id,
            **ef_w, **h_w, **t_w,
        })

    return pd.DataFrame(rows)


In [11]:

# Run only after testing one participant above.
#
ASD_features = process_group("ASD", ASD_PAIRS)
TD_features = process_group("TD", TD_PAIRS)

ASD_features.to_csv(BASE / "ASD_band_PSD_DE_features.csv", index=False)
TD_features.to_csv(BASE / "TD_band_PSD_DE_features.csv", index=False)


ASD: EF 1170 / ToM 3200
ASD: EF 1161 / ToM 3081
ASD: EF 1121 / ToM 3041
ASD: EF 1101 / ToM 3061


KeyboardInterrupt: 


## Before CCA

The wide tables are deliberately high-dimensional. With the current matched sample, do **not** run ordinary CCA yet. After preprocessing more participants, the next step should be feature QC, standardization, and either dimensionality reduction or regularized CCA.
